# M2 · Tools & Function Calling

> **Goal:** give your agent *hands*. Define function tools, watch the agent loop
> actually iterate, and add a human **approval gate** for sensitive actions.
>
> **You'll use:** the `@tool` decorator, `Agent(tools=[...])`, and approval modes.

---

A model alone can only *talk*. **Tools** let an agent *act* — look up data, call an
API, do math, hit a database. When you attach tools, the agent loop comes alive:

1. The model decides it needs a tool and emits a **tool call**.
2. The framework runs your Python function and feeds the **result** back.
3. The model continues — possibly calling more tools — until it has a final answer.

## 1. Setup

In [ ]:
# Make workshop_utils importable and pick the model (same in every lab).
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))
from workshop_utils import get_chat_client
from agent_framework import Agent, tool
from typing import Annotated
from pydantic import Field

## 2. Define a tool

A tool is just a Python function with:

- **type-annotated parameters** (so the model knows what to pass),
- a **docstring** (so the model knows *when* to use it),
- the `@tool` decorator.

`Annotated[..., Field(description=...)]` gives the model a clear description of
each argument.

!!! warning "`approval_mode` in production"
    We use `approval_mode="never_require"` here for brevity. For any tool with
    real side effects (sending email, spending money, deleting data) use
    `approval_mode="always_require"` — shown in section 5.

In [ ]:
from random import randint

@tool(approval_mode="never_require")
def get_weather(
    location: Annotated[str, Field(description="The location to get the weather for.")],
) -> str:
    """Get the weather for a given location."""
    conditions = ["sunny", "cloudy", "rainy", "stormy"]
    return f"The weather in {location} is {conditions[randint(0, 3)]} with a high of {randint(10, 30)}°C."

print("tool defined:", get_weather)

## 3. Attach the tool to an agent

Pass tools in a list. The instructions tell the agent it's allowed (and expected)
to use them.

In [ ]:
agent = Agent(
    client=get_chat_client(),
    name="WeatherAgent",
    instructions="You are a helpful weather agent. Use the get_weather tool to answer questions.",
    tools=[get_weather],
)

result = await agent.run("What's the weather like in Seattle?")
print(result)

!!! note "The loop iterated"
    To answer that, the agent made **two** model calls: one to decide *"I should
    call `get_weather('Seattle')`"*, and one to turn the tool's result into a
    sentence. You wrote a normal Python function; the framework handled the loop.

## 4. Multiple tools

Give the agent a couple of tools and it will pick the right one (or several) per
question. Real agents are mostly *tool design* — small, well-described, single-purpose
functions.

In [ ]:
@tool(approval_mode="never_require")
def convert_currency(
    amount: Annotated[float, Field(description="Amount of money to convert.")],
    to_currency: Annotated[str, Field(description="ISO code to convert to, e.g. EUR, JPY.")],
) -> str:
    """Convert an amount from US dollars to another currency (demo rates)."""
    rates = {"EUR": 0.92, "JPY": 157.0, "GBP": 0.79, "INR": 83.2}
    rate = rates.get(to_currency.upper())
    if rate is None:
        return f"Sorry, I don't have a rate for {to_currency}."
    return f"${amount:.2f} is about {amount * rate:.2f} {to_currency.upper()}."

multi_agent = Agent(
    client=get_chat_client(),
    name="TravelHelper",
    instructions="You help travelers. Use tools for weather and currency questions.",
    tools=[get_weather, convert_currency],
)

print(await multi_agent.run("I'm visiting Tokyo. What's the weather, and what is $100 in yen?"))

## 5. Human-in-the-loop: tool approval

For risky actions, require a human to approve each call. Set
`approval_mode="always_require"`. The run then **pauses** and returns
*user-input requests* instead of silently executing.

Below is a minimal approval loop: run, check for requests, approve/deny, resume.
(In a real app the approval would come from a UI; here we auto-approve to keep the
lab non-interactive — flip `APPROVE` to `False` to see a denial.)

In [ ]:
from agent_framework import Message

@tool(approval_mode="always_require")
def transfer_funds(
    amount: Annotated[float, Field(description="Amount to transfer in USD.")],
    to_account: Annotated[str, Field(description="Destination account id.")],
) -> str:
    """Transfer money between accounts. Sensitive — requires approval."""
    return f"✅ Transferred ${amount:.2f} to account {to_account}."

bank_agent = Agent(
    client=get_chat_client(),
    name="BankAgent",
    instructions="You help with banking. Use transfer_funds to move money.",
    tools=[transfer_funds],
)

APPROVE = True  # flip to False to reject the action

query = "Please transfer $50 to account ACC-12345."
result = await bank_agent.run(query)

# The agent pauses on the sensitive tool and asks for approval.
while result.user_input_requests:
    new_inputs = [query]
    for req in result.user_input_requests:
        print(f"⏸  Approval requested: {req.function_call.name}({req.function_call.arguments})")
        new_inputs.append(Message("assistant", [req]))
        new_inputs.append(Message("user", [req.to_function_approval_response(APPROVE)]))
    result = await bank_agent.run(new_inputs)

print("Agent:", result)

!!! tip "This is your first taste of *agent harness* machinery"
    Approval gates, the tool loop, retries — these are exactly the kinds of
    cross-cutting concerns the **agent harness** (M4) bundles for you. You're
    building them by hand now so the harness makes sense later.

## 🧪 Your turn

1. Add a third tool (e.g. `get_time(timezone)` returning a fixed string) and ask a
   question that needs all three.
2. Make `convert_currency` raise an exception for an unknown currency *instead* of
   returning a message, and observe how the agent reacts. (Agent Framework can
   recover from tool failures — see the upstream
   `function_tool_recover_from_failures.py`.)
3. Switch `transfer_funds` approval to a real prompt with
   `input("approve? ")` and try denying it.

---

✅ **Your agent can act now.** Next: control what it *remembers*.
→ **[M3 · Context Engineering](../03-context-engineering/)**